# 12 - Gold 1: conformed dimensions
**Tables:** `dim_region`, `dim_tower`, `dim_date`

**Surrogate-key policy:** `region_key` and `tower_key` are `xxhash64(business key)`. The same business key always gives the same
number, so keys stay stable when you re-run, and facts never need a lookup to a changing sequence.

**Join rule:** facts join to the dimensions on `region_id` and `tower_id` (the business keys), then carry the surrogate key.

In [0]:
BRONZE = "capstone6_telecom.bronze"
SILVER = "capstone6_telecom.silver"
GOLD   = "capstone6_telecom.gold"
cat    = GOLD   # this notebook writes to the gold schema

## 1. dim_region and dim_tower

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {cat}.dim_region AS
SELECT xxhash64(region_id) AS region_key, region_id, display_name, region_group,
       is_active, effective_date, risk_tier
FROM {SILVER}.silver_dim_region""")

spark.sql(f"""
CREATE OR REPLACE TABLE {cat}.dim_tower AS
SELECT xxhash64(tower_id) AS tower_key, tower_id, display_name, region_group,
       is_active, effective_date, risk_tier
FROM {SILVER}.silver_dim_tower""")

## 2. dim_date (covers every event date and every status-change date)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {cat}.dim_date AS
WITH rng AS (
  SELECT MIN(d) AS d0, MAX(d) AS d1 FROM (
    SELECT CAST(event_ts AS DATE) AS d FROM {SILVER}.silver_network_events
    UNION ALL
    SELECT CAST(change_ts AS DATE) FROM {SILVER}.silver_status_history)
), days AS (
  SELECT explode(sequence(d0, d1, interval 1 day)) AS date FROM rng
)
SELECT CAST(date_format(date, 'yyyyMMdd') AS INT) AS date_key, date,
       year(date) AS year, month(date) AS month, date_format(date, 'MMMM') AS month_name,
       day(date) AS day_of_month, date_format(date, 'EEEE') AS day_name,
       dayofweek(date) IN (1, 7) AS is_weekend
FROM days""")

## 3. Checks (expect 110 regions, 150 towers, and 0 duplicate keys)

In [0]:
display(spark.sql(f"""
SELECT 'dim_region' AS tbl, COUNT(*) AS rows, COUNT(DISTINCT region_id) AS unique_business_keys,
       COUNT(DISTINCT region_key) AS unique_surrogate_keys FROM {cat}.dim_region
UNION ALL
SELECT 'dim_tower', COUNT(*), COUNT(DISTINCT tower_id), COUNT(DISTINCT tower_key) FROM {cat}.dim_tower
UNION ALL
SELECT 'dim_date', COUNT(*), COUNT(DISTINCT date), COUNT(DISTINCT date_key) FROM {cat}.dim_date"""))

In [0]:
display(spark.table(f"{cat}.dim_date").orderBy("date").limit(5))